# 10 - Reading and Writing in Microsoft Fabric

**Suggested time: 20 minutes**

This lesson moves from notebook-created examples to files and Delta tables in a Fabric Lakehouse.

## Learning objectives

By the end of this notebook, you will be able to:

- read Lakehouse CSV files with explicit schemas;
- distinguish path-based Delta data from a managed table;
- choose between overwrite and append deliberately; and
- write and read managed Delta tables.

## Prerequisite recap and setup

Earlier notebooks created small DataFrames directly. This lesson reads the supplied files instead.

1. Use a Fabric Spark / PySpark notebook.
2. Attach a default Lakehouse.
3. Upload `orders.csv`, `customers.csv`, and `products.csv` to `Files/pyspark_training`.

Fabric provides the `spark` session. Do not stop it or replace it with a local session.

In [ ]:
input_root = 'Files/pyspark_training'
orders_path = f'{input_root}/orders.csv'
customers_path = f'{input_root}/customers.csv'
products_path = f'{input_root}/products.csv'

print(orders_path)
print(customers_path)
print(products_path)

## Read a CSV with an explicit schema

An explicit schema documents the source contract and avoids the extra scan and uncertainty of schema inference.

In [ ]:
orders_schema = '''
    order_id INT,
    customer_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DECIMAL(10,2),
    order_date DATE
'''

orders = (
    spark.read
    .option('header', True)
    .schema(orders_schema)
    .csv(orders_path)
)

orders.show()
orders.printSchema()
print('Order rows:', orders.count())

## Write Delta data to a path

`save` writes Delta files and their transaction log to a path. It does not create a named table in the Lakehouse catalogue.

In [ ]:
orders_delta_path = 'Files/pyspark_training/output/orders_delta'

(
    orders.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(orders_delta_path)
)

orders_from_delta_path = spark.read.format('delta').load(orders_delta_path)
print('Rows read from Delta path:', orders_from_delta_path.count())

## Write a managed Delta table

`saveAsTable('retail_orders')` writes Delta data and registers a named table in the attached Lakehouse. Downstream code can then use `spark.table('retail_orders')` without knowing its storage path.

In [ ]:
(
    orders.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('retail_orders')
)

retail_orders = spark.table('retail_orders')
retail_orders.show()
print('Managed-table rows:', retail_orders.count())

## Choose the write mode deliberately

- `overwrite` replaces the target and makes this training notebook safely rerunnable.
- `append` adds rows and can create duplicates when a load is rerun.
- `error` fails if the target exists.
- `ignore` leaves an existing target unchanged.

A typical append uses `new_rows.write.format('delta').mode('append').saveAsTable('retail_orders')`. Do not run that pattern here unless the rows are genuinely new.

## Your turn

Read `customers.csv` from `customers_path` with an explicit schema. Name the DataFrame `customers`, inspect it, and overwrite a managed Delta table named `retail_customers`. Read the table back as `retail_customers`.

In [ ]:
# Write your solution here.

### Expected result

The source DataFrame and managed table each contain five rows with columns `customer_id`, `customer_name`, and `region`.

### Solution - reveal after attempting

In [ ]:
customers_schema = '''
    customer_id STRING,
    customer_name STRING,
    region STRING
'''

customers = (
    spark.read
    .option('header', True)
    .schema(customers_schema)
    .csv(customers_path)
)
customers.show()
customers.printSchema()

(
    customers.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('retail_customers')
)

retail_customers = spark.table('retail_customers')
print('Customer rows:', retail_customers.count())

## Check the third capstone file

The afternoon problem also uses the product lookup. Reading it now confirms that all three inputs are ready.

In [ ]:
products_schema = 'product_id STRING, product_name STRING, category STRING'
products = spark.read.option('header', True).schema(products_schema).csv(products_path)
products.show()
print('Product rows:', products.count())

## Afternoon pipeline checklist

A typical batch solution now follows a familiar path:

1. **Read** the three Lakehouse CSV files with explicit schemas - notebook 10.
2. **Inspect and type** each dataset - notebooks 02 and 03.
3. **Clean and derive** fields such as order value - notebooks 04, 05, and 08.
4. **Join** orders to customer and product lookups - notebook 07.
5. **Aggregate and sort** the requested business result - notebook 06.
6. **Choose a latest record** when the problem contains history - this notebook.
7. **Write and verify** a managed Delta result - notebook 10.

At each stage, check the schema, grain, row count, nulls, and duplicate keys.

## Key takeaway

Use Lakehouse-relative paths for attached data, define schemas explicitly, and choose both the Delta target type and write mode deliberately.

You have completed the guided course and are ready for the Delta Lake exercise.